# BM25 + Vector Search for RAG

A minimal, iterative Retrieval-Augmented Generation pipeline using:

- **BM25** for keyword-based retrieval (`rank_bm25`)
- **Vector search** for semantic retrieval (FAISS + `text-embedding-3-small`)
- **Hybrid fusion** via Reciprocal Rank Fusion (RRF)
- **Answer generation** with `gpt-4o-mini`

The final cell runs an interactive loop where you can type questions about your document until you type `exit`.

## 1. Install Dependencies

In [1]:
!pip install -q openai faiss-cpu rank_bm25 pypdf langchain langchain-text-splitters numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 8.0 MB/s eta 0:00:00


## 2. Imports and API Key

In [3]:
import os
import re
import numpy as np
import faiss

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from openai import OpenAI
from google.colab import files, userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536
LLM_MODEL   = "gpt-4o-mini"

print("OpenAI key loaded :", bool(os.environ.get("OPENAI_API_KEY")))
print("Embedding model   :", EMBED_MODEL)
print("LLM model         :", LLM_MODEL)

OpenAI key loaded : True
Embedding model   : text-embedding-3-small
LLM model         : gpt-4o-mini


## 3. Upload a PDF

In [4]:
uploaded  = files.upload()
pdf_path  = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}  ({os.path.getsize(pdf_path):,} bytes)")

Saving The_Brain_behind_AI (1).pdf to The_Brain_behind_AI (1).pdf
Uploaded: The_Brain_behind_AI (1).pdf  (43,374 bytes)


## 4. Parse and Chunk the PDF

The document is split into overlapping chunks of 500 characters. Each chunk stores its page number so we can cite sources in the final answer.

In [5]:
def load_pdf(path):
    reader = PdfReader(path)
    pages  = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": i, "text": text})
    print(f"[LOAD] {len(pages)} non-empty pages extracted")
    return pages

def build_chunks(pages, size=500, overlap=60):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for p in pages:
        for piece in splitter.split_text(p["text"]):
            chunks.append({"id": len(chunks), "page": p["page"], "text": piece})
    print(f"[CHUNK] {len(chunks)} chunks created  (size={size}, overlap={overlap})")
    return chunks

pages  = load_pdf(pdf_path)
chunks = build_chunks(pages)

if not chunks:
    raise RuntimeError("No text extracted. Try a different PDF.")

print(f"\nFirst chunk preview:")
print(chunks[0]["text"][:300])

[LOAD] 15 non-empty pages extracted
[CHUNK] 119 chunks created  (size=500, overlap=60)

First chunk preview:
The Brain behind AI
Author: AI Generated eBook


## 5. Build the BM25 Index

We tokenize each chunk (lowercase alphanumerics) and fit a `BM25Okapi` index. This takes a fraction of a second regardless of document size.

In [6]:
def tokenize(text):
    return re.findall(r"[A-Za-z0-9]+", text.lower())

corpus_tokens = [tokenize(c["text"]) for c in chunks]
bm25          = BM25Okapi(corpus_tokens)

print(f"[BM25] Index built over {len(corpus_tokens)} chunks")
print(f"[BM25] Vocabulary size: {len(bm25.idf)} unique terms")

[BM25] Index built over 119 chunks
[BM25] Vocabulary size: 2190 unique terms


## 6. Embed Chunks and Build the FAISS Index

Each chunk is embedded with `text-embedding-3-small`, normalized to unit length, and added to a `IndexFlatIP` (exact cosine similarity). For a typical PDF this takes under a minute.

In [7]:
def embed(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i+batch])
        out.extend([d.embedding for d in resp.data])
        print(f"[EMB] {min(i+batch, len(texts))}/{len(texts)} embedded")
    arr = np.array(out, dtype="float32")
    arr /= np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12
    return arr

print("Embedding corpus...")
xb          = embed([c["text"] for c in chunks])
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(xb)
print(f"[FAISS] Index ready  ntotal={faiss_index.ntotal}  shape={xb.shape}")

Embedding corpus...
[EMB] 64/119 embedded
[EMB] 119/119 embedded
[FAISS] Index ready  ntotal=119  shape=(119, 1536)


## 7. Retrieval Functions

Three functions:

- `bm25_search` - keyword retrieval
- `vector_search` - semantic retrieval
- `hybrid_search` - combines both via RRF

In [8]:
# --- BM25 ---
def bm25_search(query, k=10):
    scores  = bm25.get_scores(tokenize(query))
    ranked  = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]
    return [{"rank": r+1, "doc_id": i, "score": float(s), "text": chunks[i]["text"], "page": chunks[i]["page"]}
            for r, (i, s) in enumerate(ranked)]

# --- Vector ---
def vector_search(query, k=10):
    q = embed([query])
    D, I = faiss_index.search(q, k)
    return [{"rank": r+1, "doc_id": int(i), "score": float(s), "text": chunks[int(i)]["text"], "page": chunks[int(i)]["page"]}
            for r, (i, s) in enumerate(zip(I[0], D[0])) if i != -1]

# --- RRF fusion ---
def rrf_fuse(lists, k=5, k_rrf=60):
    scores = {}
    texts  = {}
    for ranked in lists:
        for item in ranked:
            did = item["doc_id"]
            scores[did] = scores.get(did, 0.0) + 1.0 / (k_rrf + item["rank"])
            texts[did]  = item
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [{"rank": r+1, "doc_id": did, "rrf_score": scores[did],
             "text": texts[did]["text"], "page": texts[did]["page"]}
            for r, did in enumerate(top)]

# --- Hybrid wrapper ---
def hybrid_search(query, k=5, candidates=15):
    bm25_hits = bm25_search(query, k=candidates)
    vec_hits  = vector_search(query, k=candidates)
    return rrf_fuse([bm25_hits, vec_hits], k=k)

print("Retrieval functions ready.")

Retrieval functions ready.


## 8. Answer Generation

Takes the fused chunks as context and calls `gpt-4o-mini` with a strict grounding prompt. Page numbers are included so the model can cite them.

In [9]:
def generate_answer(query, retrieved):
    context = "\n\n---\n\n".join(
        f"[page {r['page']}, doc {r['doc_id']}]\n{r['text']}"
        for r in retrieved
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system",
             "content": ("You are a precise assistant. Answer using only the provided context. "
                         "If the answer is not in the context, say you do not know. "
                         "Cite page numbers in square brackets.")},
            {"role": "user",
             "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        temperature=0.1,
    )
    return resp.choices[0].message.content

print("Answer function ready.")

Answer function ready.


## 9. Single Query Demo

Run one query end-to-end and print the retrieved chunks so you can see what the model is working with.

In [10]:
DEMO_QUERY = "What is this document mainly about?"

print(f"Query: {DEMO_QUERY!r}")
print("-" * 70)

hits = hybrid_search(DEMO_QUERY, k=5)

print(f"Top {len(hits)} chunks retrieved by hybrid search:\n")
for h in hits:
    print(f"  [rank {h['rank']}]  page={h['page']}  rrf={h['rrf_score']:.5f}  doc_id={h['doc_id']}")
    print(f"  {h['text'][:250]}{'...' if len(h['text'])>250 else ''}")
    print()

print("-" * 70)
answer = generate_answer(DEMO_QUERY, hits)
print("Answer:\n")
print(answer)

Query: 'What is this document mainly about?'
----------------------------------------------------------------------
[EMB] 1/1 embedded
Top 5 chunks retrieved by hybrid search:

  [rank 1]  page=14  rrf=0.02991  doc_id=106
  fidelity that it **passes the cognitive Turing test** in narrow, but profound, domains. This is the heart of
Chapter 5: not to declare that AI has achieved human-like understanding, but to ask — *what does the
success of transformers reveal about the...

  [rank 2]  page=4  rrf=0.01639  doc_id=15
  2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google
Translate’s shift from phrase-based to neural models foreshadowed the transformer era.* --- ###
**Chapter 2: The Anatomy of Attention — How Transformer...

  [rank 3]  page=14  rrf=0.01613  doc_id=102
  Chapter 5
# **Chapter 5: The Next Mind — Toward Artificial General Intelligence** ### *What Transformers Tell
Us About Human Cognition and the Future of AI* > *"We are not bu

In [12]:
print(f"Full content of the top {len(hits)} retrieved chunks:\n")
for h in hits:
    print(f"--- [Rank {h['rank']}] Page {h['page']} (Doc ID: {h['doc_id']}) ---")
    print(h['text'])
    print("\n")

Full content of the top 5 retrieved chunks:

--- [Rank 1] Page 14 (Doc ID: 106) ---
fidelity that it **passes the cognitive Turing test** in narrow, but profound, domains. This is the heart of
Chapter 5: not to declare that AI has achieved human-like understanding, but to ask — *what does the
success of transformers reveal about the nature of intelligence itself?* And more urgently: **Are we on
the cusp of something greater — or merely scaling the wrong kind of mind?** --- ### **II. The Cognitive


--- [Rank 2] Page 4 (Doc ID: 15) ---
2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google
Translate’s shift from phrase-based to neural models foreshadowed the transformer era.* --- ###
**Chapter 2: The Anatomy of Attention — How Transformers See the World** *Decoding the Core
Mechanism That Changed Everything* Here, we dissect the transformer’s revolutionary engine:
**self-attention**. Through intuitive visuals and step-by-step explanations, we 

### Comparison: BM25 vs. Vector Search
Let's look at what each algorithm found before they were fused together.

In [13]:
bm25_results = bm25_search(DEMO_QUERY, k=5)
vector_results = vector_search(DEMO_QUERY, k=5)

print(f"QUERY: {DEMO_QUERY}\n")
print(f"{'[ BM25 (Keyword) ]':<45} | {'[ Vector (Semantic) ]':<45}")
print("-" * 95)

for i in range(5):
    b_text = bm25_results[i]['text'][:40].replace('\n', ' ') + "..."
    v_text = vector_results[i]['text'][:40].replace('\n', ' ') + "..."
    print(f"Rank {i+1}: {b_text:<38} (p.{bm25_results[i]['page']}) | {v_text:<38} (p.{vector_results[i]['page']})")

[EMB] 1/1 embedded
QUERY: What is this document mainly about?

[ BM25 (Keyword) ]                            | [ Vector (Semantic) ]                        
-----------------------------------------------------------------------------------------------
Rank 1: fidelity that it **passes the cognitive ... (p.14) | 2017 “Attention Is All You Need” paper, ... (p.4)
Rank 2: Chapter 5 # **Chapter 5: The Next Mind —... (p.14) | Chapter 3 # **Chapter 3: From Language t... (p.10)
Rank 3: real. But so is the need for **responsib... (p.13) | Chapter 2 # **Chapter 2: The Anatomy of ... (p.8)
Rank 4: chatbot that turned toxic—and how transf... (p.4) | technical manual, nor a dystopian prophe... (p.2)
Rank 5: This forest is *artificial intelligence*... (p.2) | *describe* it: > *“There is a 2.1 cm nod... (p.10)


## 10. Iterative QA Loop

Type any question about the document and press Enter to get an answer. The retrieved chunks are printed before the answer so you can trace the reasoning. Type `exit` to stop.

In [ ]:
print("RAG loop started. Type your question or 'exit' to stop.")
print("=" * 70)

while True:
    query = input("\nYour question: ").strip()

    if not query:
        print("Please enter a question.")
        continue

    if query.lower() in ("exit", "quit", "q"):
        print("Loop ended.")
        break

    print(f"\n[RETRIEVE] Running BM25 + Vector search...")
    hits = hybrid_search(query, k=5)

    print(f"[RETRIEVE] Top {len(hits)} chunks:")
    for h in hits:
        print(f"  rank {h['rank']}  page={h['page']}  rrf={h['rrf_score']:.5f}  "
              f"-> {h['text'][:120]}...")

    print(f"\n[GENERATE] Calling {LLM_MODEL}...")
    answer = generate_answer(query, hits)

    print("\nAnswer:")
    print("-" * 70)
    print(answer)
    print("-" * 70)

## How It Works

```
User query
    |
    +---> BM25 search  (top-15 by keyword score)  ---+
    |                                                 |
    +---> Vector search (top-15 by cosine score) ---+---> RRF fusion (top-5)
                                                              |
                                                       gpt-4o-mini
                                                              |
                                                         Final answer
```

| Component     | Role                                             | Why it is here               |
|---------------|--------------------------------------------------|------------------------------|
| BM25          | Exact keyword match, rare terms, codes, names    | Vector misses exact terms     |
| Vector (FAISS)| Semantic match, paraphrases, synonyms            | BM25 misses rephrased queries |
| RRF           | Score-agnostic fusion of both ranked lists       | Simple, robust, no tuning    |
| gpt-4o-mini   | Grounded answer generation from retrieved chunks | Synthesis and citation        |